# preprocess_dataset

 This creates the processed folder structure, splits valid pairs into train / val / test, and saves `data/processed/metadata.csv`.

## Imports and project paths
This cell loads dependencies and defines portable project-relative paths for Jupyter.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()

    for p in [start, *start.parents]:
        if (p / "data").exists() and (p / "src").exists():
            return p

    for p in [start, *start.parents]:
        if (p / "README.md").exists():
            return p

    return start

PROJECT_ROOT = find_project_root()
VALID_PAIRS_PATH = PROJECT_ROOT / "outputs" / "inspection" / "valid_pairs.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_DIR = PROCESSED_DIR / "train"
VAL_DIR = PROCESSED_DIR / "val"
TEST_DIR = PROCESSED_DIR / "test"

def to_rel_str(p):
    p = Path(p)
    try:
        return str(p.resolve().relative_to(PROJECT_ROOT.resolve())).replace("\\", "/")
    except Exception:
        return str(p).replace("\\", "/")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("VALID_PAIRS_PATH:", VALID_PAIRS_PATH)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_ROOT: C:\Users\Rajesh\Desktop\MSc-II\AI_FOR_SPACE\flood_project
VALID_PAIRS_PATH: C:\Users\Rajesh\Desktop\MSc-II\AI_FOR_SPACE\flood_project\outputs\inspection\valid_pairs.csv
PROCESSED_DIR: C:\Users\Rajesh\Desktop\MSc-II\AI_FOR_SPACE\flood_project\data\processed


## Create processed split folders

In [2]:
for split_dir in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    (split_dir / "pre").mkdir(parents=True, exist_ok=True)
    (split_dir / "post").mkdir(parents=True, exist_ok=True)
    (split_dir / "mask").mkdir(parents=True, exist_ok=True)

print("Created/verified train, val, test folders.")


Created/verified train, val, test folders.


## Load valid pairs and prepare split metadata

In [3]:
df = pd.read_csv(VALID_PAIRS_PATH)
print("Loaded valid pairs:", len(df))

keep_cols = ["pre_path", "post_path", "label_path"]
df = df[keep_cols].copy()

for col in keep_cols:
    df[col] = df[col].apply(to_rel_str)

df["sample_id"] = [f"sample_{i:05d}" for i in range(len(df))]

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, shuffle=True)

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

final_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
final_df = final_df[["sample_id", "split", "pre_path", "post_path", "label_path"]]

print("Train samples:", len(train_df))
print("Val samples:", len(val_df))
print("Test samples:", len(test_df))

metadata_path = PROCESSED_DIR / "metadata.csv"
final_df.to_csv(metadata_path, index=False)

print("Saved metadata to:", metadata_path)
display(final_df.head())


Loaded valid pairs: 202
Train samples: 141
Val samples: 30
Test samples: 31
Saved metadata to: C:\Users\Rajesh\Desktop\MSc-II\AI_FOR_SPACE\flood_project\data\processed\metadata.csv


,sample_id,split,pre_path,post_path,label_path
0,sample_00097,train,src/data/raw/PRE-event/10500500C4DD7000_0_28_6...,src/data/raw/POST-event/1040050035DC3B00_0_28_...,src/data/raw/annotations/0_28_62.geojson
1,sample_00031,train,src/data/raw/PRE-event/10500500C4DD7000_0_39_6...,src/data/raw/POST-event/10500500E6DD3C00_0_39_...,src/data/raw/annotations/0_39_67.geojson
2,sample_00012,train,src/data/raw/PRE-event/10500500C4DD7000_0_17_6...,src/data/raw/POST-event/10500500E6DD3C00_0_17_...,src/data/raw/annotations/0_17_66.geojson
3,sample_00035,train,src/data/raw/PRE-event/10500500C4DD7000_0_25_7...,src/data/raw/POST-event/10500500E6DD3C00_0_25_...,src/data/raw/annotations/0_25_70.geojson
4,sample_00119,train,src/data/raw/PRE-event/10500500C4DD7000_0_37_6...,src/data/raw/POST-event/10500500E6DD3C00_0_37_...,src/data/raw/annotations/0_37_69.geojson
